# VietNews Dataset Analytics — Google Colab (GPU)

Phân tích **toàn bộ** dataset `nam194/vietnews` (train + validation + test) trên Colab.

**Output:** `analytics/*.json` + `charts/*.png` → zip `vietnews_analytics_colab.zip`

**Import local:** `python scripts/import_colab_analytics.py vietnews_analytics_colab.zip`


In [ ]:
# Cell 1 — Cài dependencies
!pip install -q datasets matplotlib seaborn wordcloud transformers tqdm rouge-score numpy

import os, sys, re, json, math, random, hashlib, time, shutil, zipfile
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean, median, pstdev
from typing import Any, Callable, Iterator

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

IS_COLAB = "google.colab" in sys.modules
print("Colab:", IS_COLAB)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 — Cấu hình
DATASET_NAME = "nam194/vietnews"
DEFAULT_MODEL = "VietAI/vit5-base"
MAX_INPUT_TOKENS = 1024
MAX_TARGET_TOKENS = 256
LOAD_BATCH_SIZE = 2000
TOKEN_BATCH_SIZE = 128 if DEVICE == "cuda" else 64
SCATTER_VIZ_MAX = 5000
CORR_VIZ_MAX = 10000
ROUGE_FULL_MAX = 5000
NGRAM_FULL_MAX = 150000

WORKDIR = Path("/content/vietnews_analytics")
ANALYTICS_DIR = WORKDIR / "analytics"
CHARTS_DIR = WORKDIR / "charts"
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", WORKDIR)


In [ ]:
# Cell 3 — Tiền xử lý văn bản (core logic từ preprocess/preprocessor.py)
VI_LETTER_RE = r"A-Za-zÀ-ỹĐđ"
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\u202a-\u202e\ufeff]")
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z0-9#]+;")
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b\S+@\S+\.\S+\b")
TOKEN_RE = re.compile(rf"[{VI_LETTER_RE}0-9]+", re.UNICODE)

VN_STOPWORDS = {
    "và", "của", "các", "những", "một", "trong", "cho", "với", "được", "đã",
    "là", "có", "không", "này", "đó", "từ", "khi", "về", "theo", "sau",
    "trước", "tại", "để", "nhiều", "người", "năm", "ngày", "ra", "vào",
}


def normalize_unicode(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text) if "unicodedata" in dir() else text
    import unicodedata
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = CONTROL_CHARS_RE.sub(" ", text)
    return text


def remove_html_tags(text: str) -> str:
  import html as html_mod
  text = html_mod.unescape(text or "")
  text = re.sub(r"<[^>]+>", " ", text)
  text = HTML_ENTITY_RE.sub(" ", text)
  return text


def normalize_whitespace(text: str) -> str:
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def normalize_punctuation(text: str) -> str:
    text = re.sub(r"([!?]){2,}", r"\1", text)
    text = re.sub(r"([,;:]){2,}", r"\1", text)
    text = re.sub(r"\.{3,}", "...", text)
    text = re.sub(r"\s*([,;:!?])\s*", r"\1 ", text)
    return text


def remove_noise_characters(text: str) -> str:
    return re.sub(rf"([^{VI_LETTER_RE}\s0-9.,!?;:'\"()/%+-]){{2,}}", " ", text, flags=re.UNICODE)


def clean_text(text: str, aggressive: bool = False) -> str:
    if not text:
        return ""
    text = normalize_unicode(text)
    text = remove_html_tags(text)
    if aggressive:
        text = URL_RE.sub(" ", text)
        text = EMAIL_RE.sub(" ", text)
    text = normalize_punctuation(text)
    text = remove_noise_characters(text)
    text = normalize_whitespace(text)
    return text


def clean_article(text: str) -> str:
    return clean_text(text or "", aggressive=True)


def clean_summary(text: str) -> str:
    return clean_text(text or "", aggressive=False)


def split_sentences(text: str) -> list[str]:
    text = clean_text(text) if text else ""
    if not text:
        return []
    pieces = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in pieces if p and p.strip()]


def tokenize_words(text: str, remove_stopwords: bool = False) -> list[str]:
    import unicodedata
    norm = unicodedata.normalize("NFC", text or "")
    tokens = [m.group(0).lower() for m in TOKEN_RE.finditer(norm)]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in VN_STOPWORDS and len(t) > 1]
    return tokens


def text_fingerprint(text: str) -> str:
    normalized = " ".join(tokenize_words(clean_text(text), remove_stopwords=False))
    return hashlib.sha1(normalized.encode("utf-8")).hexdigest()

print("Text utilities ready")


In [ ]:
# Cell 4 — Load FULL dataset nam194/vietnews
from datasets import load_dataset

@dataclass
class DatasetRecord:
    article: str
    abstract: str
    title: str = ""
    category: str = ""
    guid: str = ""
    split: str = ""


@dataclass
class LoadedDataset:
    dataset_name: str
    records: list = field(default_factory=list)
    splits: dict = field(default_factory=dict)
    columns: list = field(default_factory=list)
    source: str = "huggingface"
    limit_per_split: int | None = None
    total_raw_samples: int = 0
    split_raw_counts: dict = field(default_factory=dict)


def _row_to_record(row: dict, split: str) -> DatasetRecord:
    return DatasetRecord(
        article=str(row.get("article", "") or "").strip(),
        abstract=str(row.get("abstract", "") or "").strip(),
        title=str(row.get("title", "") or "").strip(),
        category=str(row.get("category", row.get("topic", "")) or "").strip(),
        guid=str(row.get("guid", row.get("id", "")) or "").strip(),
        split=split,
    )


def load_full_vietnews(name: str = DATASET_NAME) -> LoadedDataset:
    raw = None
    for kwargs in ({}, {"trust_remote_code": True}):
        try:
            raw = load_dataset(name, **kwargs)
            break
        except Exception as exc:
            print("load_dataset retry:", exc)
    if raw is None:
        raise RuntimeError(f"Cannot load {name}")

    records, splits, split_raw_counts = [], {}, {}
    total_raw = 0
    columns = list(raw[list(raw.keys())[0]].column_names)
    split_names = [s for s in ("train", "validation", "test") if s in raw]

    for split_name in tqdm(split_names, desc="Loading splits"):
        split_ds = raw[split_name]
        split_raw_counts[split_name] = len(split_ds)
        total_raw += len(split_ds)
        taken = 0
        cap = len(split_ds)
        while taken < cap:
            end = min(taken + LOAD_BATCH_SIZE, cap)
            batch_rows = split_ds[taken:end]
            if isinstance(batch_rows, dict):
                keys = list(batch_rows.keys())
                n_rows = len(batch_rows[keys[0]]) if keys else 0
                for i in range(n_rows):
                    row = {k: batch_rows[k][i] for k in keys}
                    rec = _row_to_record(row, split_name)
                    if rec.article and rec.abstract:
                        records.append(rec)
            else:
                for item in batch_rows:
                    rec = _row_to_record(item, split_name)
                    if rec.article and rec.abstract:
                        records.append(rec)
            taken = end
        splits[split_name] = sum(1 for r in records if r.split == split_name)

    print(f"Loaded {len(records):,} valid pairs from {total_raw:,} raw rows")
    print("Splits:", splits)
    return LoadedDataset(
        dataset_name=name,
        records=records,
        splits=splits,
        columns=columns,
        source="huggingface",
        limit_per_split=None,
        total_raw_samples=total_raw,
        split_raw_counts=split_raw_counts,
    )

t0_load = time.perf_counter()
loaded = load_full_vietnews()
print(f"Load time: {time.perf_counter() - t0_load:.1f}s")


In [ ]:
# Cell 5 — Thống kê (tương đương backend/services/dataset_analysis/statistics.py)
from rouge_score import rouge_scorer

_ROUGE = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL", "rougeLsum"], use_stemmer=False)


def _numeric_stats(values):
    if not values:
        return {}
    vals = [float(v) for v in values]
    vals_sorted = sorted(vals)
    n = len(vals)
    def _pct(p):
        return round(vals_sorted[int(p * (n - 1))], 4) if n > 1 else round(vals_sorted[0], 4)
    return {
        "count": n, "min": round(min(vals), 4), "max": round(max(vals), 4),
        "mean": round(mean(vals), 4), "median": round(median(vals), 4),
        "std": round(pstdev(vals), 4) if n > 1 else 0.0,
        "p25": _pct(0.25), "p75": _pct(0.75), "p95": _pct(0.95), "p99": _pct(0.99),
    }


def _histogram(values, bins=30):
    if not values:
        return {"bins": [], "counts": []}
    counts, edges = np.histogram(values, bins=bins)
    return {"bins": [round(float(x), 2) for x in edges.tolist()], "counts": [int(c) for c in counts.tolist()]}


def _lead_baseline_summary(article, target_words, lead_sentences=None):
    sentences = split_sentences(article)
    if lead_sentences is not None:
        return " ".join(sentences[:lead_sentences])
    words = []
    for sent in sentences:
        words.extend(tokenize_words(sent))
        if len(words) >= target_words:
            break
    return " ".join(words[: max(target_words, 1)])


def _extractive_coverage(article, summary):
    art = set(tokenize_words(article))
    summ = tokenize_words(summary)
    return (sum(1 for w in summ if w in art) / len(summ)) if summ else 0.0


def _extractive_density(article, summary):
    art_tokens = tokenize_words(article)
    summ_tokens = tokenize_words(summary)
    if not art_tokens or not summ_tokens:
        return 0.0
    art_set = set(art_tokens)
    return sum(1 for w in summ_tokens if w in art_set) / len(art_tokens)


def _novel_ngram_pct(article, summary, n=1):
    art_tokens = tokenize_words(article)
    summ_tokens = tokenize_words(summary)
    if not summ_tokens:
        return 0.0
    if n == 1:
        art_set = set(art_tokens)
        return sum(1 for w in summ_tokens if w not in art_set) / len(summ_tokens)
    art_ngrams = {tuple(art_tokens[i:i+n]) for i in range(max(0, len(art_tokens)-n+1))}
    summ_ngrams = [tuple(summ_tokens[i:i+n]) for i in range(max(0, len(summ_tokens)-n+1))]
    if not summ_ngrams:
        return 0.0
    return sum(1 for ng in summ_ngrams if ng not in art_ngrams) / len(summ_ngrams)


def _zipf_data(counter, top_n=50):
    ranked = counter.most_common()
    return [{"rank": i+1, "word": w, "frequency": f, "log_rank": round(math.log10(i+1), 4)} for i, (w, f) in enumerate(ranked[:top_n])]


def _vocab_growth(tokens_stream, checkpoints=30):
    if not tokens_stream:
        return []
    step = max(1, len(tokens_stream) // checkpoints)
    seen, curve = set(), []
    for i, tok in enumerate(tokens_stream, start=1):
        seen.add(tok)
        if i % step == 0 or i == len(tokens_stream):
            curve.append({"tokens_seen": i, "unique_vocab": len(seen)})
    return curve


def _correlation_matrix(features):
    keys = list(features.keys())
    if not keys or not features[keys[0]]:
        return {"labels": keys, "matrix": []}
    matrix = []
    for a in keys:
        row = []
        va = np.array(features[a], dtype=float)
        for b in keys:
            vb = np.array(features[b], dtype=float)
            if len(va) < 2 or len(vb) < 2:
                row.append(0.0)
            else:
                c = float(np.corrcoef(va, vb)[0, 1])
                row.append(round(c if not math.isnan(c) else 0.0, 4))
        matrix.append(row)
    return {"labels": keys, "matrix": matrix}


def _count_paragraphs(text):
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    return max(1, len(parts)) if text.strip() else 0


def compute_rouge_batch(predictions, references):
    if not predictions:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0}
    rows = []
    for p, r in zip(predictions, references):
        if not p or not r:
            rows.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0})
            continue
        scores = _ROUGE.score(clean_text(r), clean_text(p))
        rows.append({k: round(float(v.fmeasure), 4) for k, v in scores.items()})
    return {k: round(sum(row[k] for row in rows) / len(rows), 4) for k in rows[0]}


def compute_training_statistics():
    return {
        "dataset_name": DATASET_NAME,
        "max_train_samples": 5000,
        "validation_ratio": 0.1,
        "train_batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 5e-5,
        "num_epochs": 3,
        "weight_decay": 0.01,
        "warmup_steps": 500,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_target_tokens": MAX_TARGET_TOKENS,
        "default_model": DEFAULT_MODEL,
        "use_fp16": True,
        "optimizer": "AdamW",
        "scheduler": "linear_warmup",
        "source": "colab_defaults",
    }


def compute_all_statistics(loaded, seed=42):
    rng = random.Random(seed)
    records = loaded.records
    n = len(records)
    is_full = loaded.limit_per_split is None

    articles = [clean_article(r.article) for r in records]
    summaries = [clean_summary(r.abstract) for r in records]
    titles = [r.title for r in records]

    art_sentences = [len(split_sentences(a)) for a in articles]
    sum_sentences = [len(split_sentences(s)) for s in summaries]
    art_paragraphs = [_count_paragraphs(a) for a in articles]
    title_lens = [len(tokenize_words(t)) for t in titles]
    art_words = [len(tokenize_words(a)) for a in articles]
    sum_words = [len(tokenize_words(s)) for s in summaries]
    art_chars = [len(a) for a in articles]
    sum_chars = [len(s) for s in summaries]
    art_tokens = art_words  # word-level proxy for speed
    sum_tokens = sum_words
    compression = [(sw / aw if aw > 0 else 0.0) for aw, sw in zip(art_words, sum_words)]

    word_counter, bigram_counter, trigram_counter = Counter(), Counter(), Counter()
    stopword_hits, all_tokens = Counter(), []
    ngram_n = n if (is_full and n <= NGRAM_FULL_MAX) else min(n, 5000)
    ngram_indices = rng.sample(range(n), ngram_n) if ngram_n < n else list(range(n))

    for idx in tqdm(ngram_indices, desc="N-grams"):
        tokens = tokenize_words(articles[idx], remove_stopwords=False)
        all_tokens.extend(tokens)
        word_counter.update(tokens)
        for t in tokens:
            if t in VN_STOPWORDS:
                stopword_hits[t] += 1
        for i in range(len(tokens) - 1):
            bigram_counter[(tokens[i], tokens[i+1])] += 1
        for i in range(len(tokens) - 2):
            trigram_counter[(tokens[i], tokens[i+1], tokens[i+2])] += 1

    vocab_size = len(word_counter)
    rare_words = sum(1 for _, c in word_counter.items() if c <= 2)

    fingerprints = [text_fingerprint(a) for a in articles]
    fp_counter = Counter(fingerprints)
    duplicates = sum(c - 1 for c in fp_counter.values() if c > 1)
    empty_articles = sum(1 for a in articles if not a.strip())
    empty_summaries = sum(1 for s in summaries if not s.strip())
    missing_title = sum(1 for t in titles if not t.strip())
    very_short_art = sum(1 for w in art_words if w < 30)
    very_long_art = sum(1 for w in art_words if w > 1500)
    very_short_sum = sum(1 for w in sum_words if w < 5)
    very_long_sum = sum(1 for w in sum_words if w > 200)
    art_mean, art_std = (mean(art_words), pstdev(art_words)) if art_words else (0, 0)
    outliers = sum(1 for w in art_words if art_std and abs(w - art_mean) > 3 * art_std)

    category_counter = Counter()
    for r in records:
        if r.category:
            category_counter[r.category] += 1

    rouge_sample_n = n if n <= ROUGE_FULL_MAX else ROUGE_FULL_MAX
    rouge_indices = rng.sample(range(n), rouge_sample_n) if n > rouge_sample_n else list(range(n))
    rouge_baselines = {"sample_size": rouge_sample_n, "full_dataset": rouge_sample_n == n}
    refs = [summaries[i] for i in rouge_indices]
    for name, fn in (
        ("lead_words_proportional", lambda i: _lead_baseline_summary(articles[i], sum_words[i] or 20)),
        ("lead_1", lambda i: _lead_baseline_summary(articles[i], sum_words[i] or 20, lead_sentences=1)),
        ("lead_3", lambda i: _lead_baseline_summary(articles[i], sum_words[i] or 20, lead_sentences=3)),
    ):
        preds = [fn(i) for i in rouge_indices]
        rouge_baselines[name] = compute_rouge_batch(preds, refs)

    coverages = [_extractive_coverage(articles[i], summaries[i]) for i in rouge_indices]
    densities = [_extractive_density(articles[i], summaries[i]) for i in rouge_indices]
    novel_u = [_novel_ngram_pct(articles[i], summaries[i], 1) for i in rouge_indices]
    novel_b = [_novel_ngram_pct(articles[i], summaries[i], 2) for i in rouge_indices]
    extractive_metrics = {
        "sample_size": rouge_sample_n,
        "avg_coverage": round(mean(coverages), 4) if coverages else 0,
        "avg_density": round(mean(densities), 4) if densities else 0,
        "avg_novel_unigram_pct": round(mean(novel_u), 4) if novel_u else 0,
        "avg_novel_bigram_pct": round(mean(novel_b), 4) if novel_b else 0,
    }

    scatter_n = min(SCATTER_VIZ_MAX, n)
    scatter_idx = rng.sample(range(n), scatter_n) if n > scatter_n else list(range(n))
    scatter_points = [{"article_words": art_words[i], "summary_words": sum_words[i], "compression_ratio": round(compression[i], 4)} for i in scatter_idx]
    if scatter_points:
        x = np.array([p["article_words"] for p in scatter_points], dtype=float)
        y = np.array([p["compression_ratio"] for p in scatter_points], dtype=float)
        if len(x) >= 2:
            slope, intercept = np.polyfit(x, y, 1)
            regression = {"slope": round(float(slope), 6), "intercept": round(float(intercept), 6), "equation": f"y = {slope:.6f}x + {intercept:.6f}", "scatter_sample_size": scatter_n}
        else:
            regression = {"scatter_sample_size": scatter_n}
    else:
        regression = {}

    corr_n = min(CORR_VIZ_MAX, n)
    corr_features = {
        "article_words": [float(art_words[i]) for i in range(corr_n)],
        "summary_words": [float(sum_words[i]) for i in range(corr_n)],
        "article_sentences": [float(art_sentences[i]) for i in range(corr_n)],
        "compression_ratio": [float(compression[i]) for i in range(corr_n)],
    }

    overview = {
        "dataset_name": loaded.dataset_name,
        "total_documents": n, "total_summaries": n,
        "splits": loaded.splits,
        "split_raw_counts": loaded.split_raw_counts,
        "total_raw_samples": loaded.total_raw_samples,
        "limit_per_split": loaded.limit_per_split,
        "full_dataset": is_full,
        "source": "colab",
        "columns": loaded.columns,
        "total_sentences_articles": sum(art_sentences),
        "total_sentences_summaries": sum(sum_sentences),
        "total_words_articles": sum(art_words),
        "total_words_summaries": sum(sum_words),
        "vocab_size": vocab_size, "unique_words": vocab_size,
        "avg_article_words": round(mean(art_words), 2) if art_words else 0,
        "avg_summary_words": round(mean(sum_words), 2) if sum_words else 0,
        "avg_article_sentences": round(mean(art_sentences), 2) if art_sentences else 0,
        "avg_summary_sentences": round(mean(sum_sentences), 2) if sum_sentences else 0,
        "avg_article_paragraphs": round(mean(art_paragraphs), 2) if art_paragraphs else 0,
        "avg_title_words": round(mean(title_lens), 2) if title_lens else 0,
        "avg_compression_ratio": round(mean(compression), 4) if compression else 0,
        "avg_reduction_pct": round(100 * (1 - mean(compression)), 2) if compression else 0,
    }

    return {
        "overview": overview,
        "document_stats": {"sentences": _numeric_stats(art_sentences), "words": _numeric_stats(art_words), "chars": _numeric_stats(art_chars), "tokens": _numeric_stats(art_tokens), "paragraphs": _numeric_stats(art_paragraphs)},
        "summary_stats": {"sentences": _numeric_stats(sum_sentences), "words": _numeric_stats(sum_words), "chars": _numeric_stats(sum_chars), "tokens": _numeric_stats(sum_tokens), "compression": _numeric_stats(compression), "title_words": _numeric_stats(title_lens)},
        "vocabulary": {
            "unique_words": vocab_size, "rare_words_count": rare_words, "rare_threshold": 2,
            "stopword_total_hits": sum(stopword_hits.values()),
            "top_stopwords": [{"word": w, "count": c} for w, c in stopword_hits.most_common(30)],
            "top_100_words": [{"word": w, "count": c} for w, c in word_counter.most_common(100)],
            "top_30_bigrams": [{"ngram": " ".join(ng), "count": c} for ng, c in bigram_counter.most_common(30)],
            "top_30_trigrams": [{"ngram": " ".join(ng), "count": c} for ng, c in trigram_counter.most_common(30)],
            "zipf": _zipf_data(word_counter), "vocab_growth": _vocab_growth(all_tokens),
            "ngram_sample_size": len(ngram_indices), "ngram_full_dataset": len(ngram_indices) == n,
        },
        "quality": {
            "duplicates": duplicates, "empty_articles": empty_articles, "empty_summaries": empty_summaries,
            "missing_titles": missing_title, "very_short_articles": very_short_art, "very_long_articles": very_long_art,
            "very_short_summaries": very_short_sum, "very_long_summaries": very_long_sum,
            "outliers_3sigma": outliers, "valid_pairs": n,
        },
        "length_distribution": {
            "article_words": _histogram(art_words), "summary_words": _histogram(sum_words),
            "article_sentences": _histogram(art_sentences), "compression_ratio": _histogram(compression),
        },
        "compression_statistics": {"overall": _numeric_stats(compression), "scatter_sample": scatter_points, "regression": regression, "scatter_sample_size": scatter_n},
        "correlation": {**_correlation_matrix(corr_features), "sample_size": corr_n},
        "word_frequency": {"top_200": [{"word": w, "count": c} for w, c in word_counter.most_common(200)], "total_tokens": len(all_tokens)},
        "category_stats": {"available": bool(category_counter), "categories": [{"name": c, "count": n} for c, n in category_counter.most_common(50)]},
        "rouge_baseline": rouge_baselines,
        "extractive_metrics": extractive_metrics,
        "training_statistics": compute_training_statistics(),
        "_articles": articles,
        "_summaries": summaries,
    }

t0_stats = time.perf_counter()
stats = compute_all_statistics(loaded)
# Cache toàn cục — Cell 6+ có thể chạy lại mà không cần pop lại từ stats
COLAB_ARTICLES = stats["_articles"]
COLAB_SUMMARIES = stats["_summaries"]
print(f"Statistics done in {time.perf_counter() - t0_stats:.1f}s — {len(COLAB_ARTICLES):,} records cached")


In [ ]:
# Cell 6 — Tokenizer stats (GPU batch nếu có CUDA)
from transformers import AutoTokenizer

_tokenizer = None

def get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        # ViT5/T5: use_fast=True gây lỗi Unigram trên Colab (transformers 4.4x+)
        try:
            _tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL, use_fast=False)
        except Exception:
            _tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
    return _tokenizer


def _percentile(sorted_vals, p):
    if not sorted_vals:
        return 0
    return sorted_vals[int(p * (len(sorted_vals) - 1))]


def batch_subword_lengths(texts, batch_size=TOKEN_BATCH_SIZE):
    """Đếm subword tokens — tokenizer luôn chạy CPU (không đưa lên CUDA)."""
    tok = get_tokenizer()
    lengths = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Subword tokens"):
        batch = texts[i : i + batch_size]
        try:
            enc = tok(batch, add_special_tokens=False, padding=False, truncation=False)
            ids = enc.get("input_ids") if isinstance(enc, dict) else enc
            for row in ids:
                lengths.append(len(row))
        except Exception:
            for text in batch:
                lengths.append(len(tok.encode(text, add_special_tokens=False)))
    return lengths


def token_stats_for_texts(texts, sample_size=None):
    if not texts:
        return {}
    sample = texts if (sample_size is None or sample_size <= 0 or sample_size >= len(texts)) else texts[:sample_size]
    word_counts = [len(tokenize_words(t)) for t in sample]
    subword_counts = batch_subword_lengths(sample)
    word_sorted = sorted(word_counts)
    sub_sorted = sorted(subword_counts)
    truncated_in = sum(1 for s in sub_sorted if s > MAX_INPUT_TOKENS)
    truncated_tgt = sum(1 for s in sub_sorted if s > MAX_TARGET_TOKENS)
    result = {
        "word_token_avg": round(mean(word_counts), 2),
        "word_token_min": min(word_counts), "word_token_max": max(word_counts),
        "word_token_p95": _percentile(word_sorted, 0.95), "word_token_p99": _percentile(word_sorted, 0.99),
        "tokenizer_model": DEFAULT_MODEL, "sample_size": len(sample), "full_dataset": len(sample) == len(texts),
        "subword_token_avg": round(mean(sub_sorted), 2),
        "subword_token_min": min(sub_sorted), "subword_token_max": max(sub_sorted),
        "subword_token_p95": _percentile(sub_sorted, 0.95), "subword_token_p99": _percentile(sub_sorted, 0.99),
        "truncation_estimate_input": {"max_tokens": MAX_INPUT_TOKENS, "would_truncate": truncated_in, "pct": round(100 * truncated_in / len(sub_sorted), 2)},
        "truncation_estimate_target": {"max_tokens": MAX_TARGET_TOKENS, "would_truncate": truncated_tgt, "pct": round(100 * truncated_tgt / len(sub_sorted), 2)},
    }
    return result

def resolve_text_corpus():
    """Lấy articles/summaries — an toàn khi chạy lại Cell 6 hoặc bỏ qua Cell 5."""
    if isinstance(globals().get("stats"), dict):
        if "_articles" in stats and "_summaries" in stats:
            return stats.pop("_articles"), stats.pop("_summaries")
    if globals().get("COLAB_ARTICLES") and globals().get("COLAB_SUMMARIES"):
        return COLAB_ARTICLES, COLAB_SUMMARIES
    if globals().get("loaded") is not None:
        arts = [clean_text(r.article) for r in loaded.records]
        sums = [clean_text(r.abstract) for r in loaded.records]
        print(f"Rebuild corpus from loaded: {len(arts):,} records")
        return arts, sums
    raise RuntimeError(
        "Thiếu dữ liệu. Chạy Cell 4 (load) + Cell 5 (statistics) trước, "
        "hoặc đảm bảo biến `loaded` còn trong session."
    )


articles, summaries = resolve_text_corpus()
if "token_statistics" not in stats:
    stats["token_statistics"] = {
        "articles": token_stats_for_texts(articles),
        "summaries": token_stats_for_texts(summaries),
        "subword_full_articles": token_stats_for_texts(articles),
    }
    print("Token statistics ready")
else:
    print("Token statistics already computed — skip (xóa stats['token_statistics'] để tính lại)")


In [ ]:
# Cell 7 — Sinh biểu đồ PNG (tương đương visualizer.py)
def _save_fig(name):
    path = CHARTS_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    return str(path)


def _hist_from_distribution(dist, title, xlabel, filename):
    bins, counts = dist.get("bins") or [], dist.get("counts") or []
    if not bins or not counts:
        return None
    centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(counts))]
    plt.figure(figsize=(8, 4.5))
    plt.bar(centers, counts, width=(bins[1]-bins[0])*0.9, color="#0ea5e9", alpha=0.85)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel("Số mẫu"); plt.grid(axis="y", alpha=0.3)
    return _save_fig(filename)


charts = {}
overview = stats.get("overview", {})
splits = overview.get("splits", {})
if splits:
    plt.figure(figsize=(6, 5))
    labels, values = list(splits.keys()), [splits[k] for k in splits]
    plt.pie(values, labels=labels, autopct="%1.1f%%", colors=["#6366f1", "#10b981", "#f59e0b"][:len(labels)], startangle=140)
    plt.title("Phân chia Train / Val / Test")
    charts["split_pie"] = _save_fig("split_pie")

length_dist = stats.get("length_distribution", {})
for key, title, xlab, fname in (
    ("article_words", "Phân phối độ dài bài viết (từ)", "Số từ", "hist_article_words"),
    ("summary_words", "Phân phối độ dài tóm tắt (từ)", "Số từ", "hist_summary_words"),
    ("compression_ratio", "Phân phối tỷ lệ nén", "Tỷ lệ nén", "hist_compression"),
):
    p = _hist_from_distribution(length_dist.get(key, {}), title, xlab, fname)
    if p:
        charts[fname] = p

vocab = stats.get("vocabulary", {})
top_words = vocab.get("top_100_words", [])[:20]
if top_words:
    plt.figure(figsize=(9, 5))
    words = [w["word"] for w in top_words][::-1]
    counts = [w["count"] for w in top_words][::-1]
    plt.barh(words, counts, color="#6366f1")
    plt.title("Top 20 từ phổ biến"); plt.xlabel("Tần suất")
    charts["bar_top_words"] = _save_fig("bar_top_words")

zipf = vocab.get("zipf", [])
if zipf:
    plt.figure(figsize=(7, 4.5))
    plt.loglog([z["rank"] for z in zipf], [z["frequency"] for z in zipf], "o-", color="#10b981", markersize=4)
    plt.title("Phân phối Zipf (log-log)"); plt.xlabel("Hạng"); plt.ylabel("Tần suất"); plt.grid(True, which="both", alpha=0.3)
    charts["zipf_line"] = _save_fig("zipf_line")

growth = vocab.get("vocab_growth", [])
if growth:
    plt.figure(figsize=(7, 4.5))
    plt.plot([g["tokens_seen"] for g in growth], [g["unique_vocab"] for g in growth], color="#0ea5e9", linewidth=2)
    plt.title("Tăng trưởng từ vựng"); plt.xlabel("Tokens"); plt.ylabel("Unique vocab"); plt.grid(alpha=0.3)
    charts["vocab_growth_line"] = _save_fig("vocab_growth_line")

comp = stats.get("compression_statistics", {})
scatter = comp.get("scatter_sample", [])
if scatter:
    plt.figure(figsize=(7, 5))
    x = [p["article_words"] for p in scatter]
    y = [p["compression_ratio"] for p in scatter]
    plt.scatter(x, y, alpha=0.35, s=12, color="#6366f1")
    reg = comp.get("regression", {})
    if reg.get("slope") is not None:
        xs = np.linspace(min(x), max(x), 50)
        plt.plot(xs, reg["slope"]*xs + reg["intercept"], color="#f43f5e", linewidth=2, label=reg.get("equation"))
        plt.legend()
    plt.title("Tỷ lệ nén vs độ dài bài viết"); plt.xlabel("Số từ bài viết"); plt.ylabel("Tỷ lệ nén"); plt.grid(alpha=0.3)
    charts["compression_scatter"] = _save_fig("compression_scatter")

corr = stats.get("correlation", {})
labels_c, matrix = corr.get("labels", []), corr.get("matrix", [])
if labels_c and matrix:
    plt.figure(figsize=(6, 5))
    arr = np.array(matrix, dtype=float)
    im = plt.imshow(arr, cmap="RdBu_r", vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046)
    plt.xticks(range(len(labels_c)), labels_c, rotation=45, ha="right")
    plt.yticks(range(len(labels_c)), labels_c)
    plt.title("Ma trận tương quan")
    for i in range(len(labels_c)):
        for j in range(len(labels_c)):
            plt.text(j, i, f"{arr[i,j]:.2f}", ha="center", va="center", fontsize=8)
    charts["correlation_heatmap"] = _save_fig("correlation_heatmap")

doc_stats, sum_stats = stats.get("document_stats", {}), stats.get("summary_stats", {})
if doc_stats.get("words") and sum_stats.get("words"):
    plt.figure(figsize=(7, 4.5))
    metrics = ["min", "mean", "median", "max"]
    x = np.arange(len(metrics)); w = 0.35
    plt.bar(x - w/2, [doc_stats["words"].get(m, 0) for m in metrics], w, label="Bài viết", color="#0ea5e9")
    plt.bar(x + w/2, [sum_stats["words"].get(m, 0) for m in metrics], w, label="Tóm tắt", color="#10b981")
    plt.xticks(x, metrics); plt.title("So sánh độ dài từ"); plt.legend(); plt.grid(axis="y", alpha=0.3)
    charts["bar_length_compare"] = _save_fig("bar_length_compare")

aw, sw = length_dist.get("article_words", {}), length_dist.get("summary_words", {})
if aw.get("bins") and sw.get("bins"):
    art_data, sum_data = [], []
    for c, lo, hi in zip(aw["counts"], aw["bins"][:-1], aw["bins"][1:]):
        art_data.extend([((lo+hi)/2)] * c)
    for c, lo, hi in zip(sw["counts"], sw["bins"][:-1], sw["bins"][1:]):
        sum_data.extend([((lo+hi)/2)] * c)
    if art_data and sum_data:
        plt.figure(figsize=(6, 4.5))
        plt.boxplot([art_data, sum_data], labels=["Bài viết", "Tóm tắt"])
        plt.title("Box plot độ dài (từ)"); plt.grid(axis="y", alpha=0.3)
        charts["box_length"] = _save_fig("box_length")

wf = stats.get("word_frequency", {}).get("top_200", [])[:80]
if wf:
    try:
        from wordcloud import WordCloud
        wc = WordCloud(width=900, height=450, background_color="white", colormap="viridis").generate_from_frequencies({w["word"]: w["count"] for w in wf})
        plt.figure(figsize=(10, 5)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off"); plt.title("Word Cloud")
        charts["wordcloud"] = _save_fig("wordcloud")
    except Exception as e:
        print("WordCloud skipped:", e)

bigrams = vocab.get("top_30_bigrams", [])[:15]
if bigrams:
    plt.figure(figsize=(9, 5))
    labels_ng = [b["ngram"] for b in bigrams][::-1]
    plt.barh(labels_ng, [b["count"] for b in bigrams][::-1], color="#a855f7")
    plt.title("Top 15 bigrams"); plt.xlabel("Tần suất")
    charts["bar_bigrams"] = _save_fig("bar_bigrams")

quality = stats.get("quality", {})
if quality:
    plt.figure(figsize=(7, 4.5))
    q_labels = ["Trùng lặp", "Rỗng art", "Rỗng sum", "Quá ngắn", "Quá dài", "Outliers"]
    q_vals = [quality.get(k, 0) for k in ("duplicates", "empty_articles", "empty_summaries", "very_short_articles", "very_long_articles", "outliers_3sigma")]
    plt.bar(q_labels, q_vals, color="#f43f5e", alpha=0.8)
    plt.title("Chất lượng dữ liệu"); plt.xticks(rotation=30, ha="right"); plt.ylabel("Số mẫu"); plt.grid(axis="y", alpha=0.3)
    charts["bar_quality"] = _save_fig("bar_quality")

print(f"Generated {len(charts)} charts")


In [ ]:
# Cell 8 — Xuất JSON (format storage/analytics/)
PIPELINE_T0 = t0_load

def _write_json(path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def _dir_size(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path.exists() else 0

total_elapsed = time.perf_counter() - PIPELINE_T0
n_records = len(loaded.records)

meta = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset_name": DATASET_NAME,
    "source": "colab",
    "full_dataset": True,
    "limit_per_split": None,
    "limit_mode": "full",
    "record_count": n_records,
    "total_samples": n_records,
    "total_raw_samples": loaded.total_raw_samples,
    "split_raw_counts": loaded.split_raw_counts,
    "analysis_duration_sec": round(total_elapsed, 2),
    "chart_count": len(charts),
    "cache_enabled": False,
    "device": DEVICE,
}

stats["metadata"] = meta

_write_json(ANALYTICS_DIR / "metadata.json", {**meta, "cache_size_bytes": _dir_size(ANALYTICS_DIR), "charts_size_bytes": _dir_size(CHARTS_DIR)})
_write_json(ANALYTICS_DIR / "dataset_overview.json", stats["overview"])
_write_json(ANALYTICS_DIR / "dataset_statistics.json", {"document_stats": stats["document_stats"], "summary_stats": stats["summary_stats"]})
_write_json(ANALYTICS_DIR / "dataset_quality.json", stats["quality"])
_write_json(ANALYTICS_DIR / "token_statistics.json", stats["token_statistics"])
_write_json(ANALYTICS_DIR / "compression_statistics.json", stats["compression_statistics"])
_write_json(ANALYTICS_DIR / "correlation.json", stats["correlation"])
_write_json(ANALYTICS_DIR / "word_frequency.json", stats["word_frequency"])
_write_json(ANALYTICS_DIR / "length_distribution.json", stats["length_distribution"])
_write_json(ANALYTICS_DIR / "training_statistics.json", stats["training_statistics"])
_write_json(ANALYTICS_DIR / "vocabulary.json", stats["vocabulary"])
_write_json(ANALYTICS_DIR / "category_stats.json", stats["category_stats"])
_write_json(ANALYTICS_DIR / "rouge_baseline.json", stats["rouge_baseline"])
_write_json(ANALYTICS_DIR / "extractive_metrics.json", stats["extractive_metrics"])
_write_json(ANALYTICS_DIR / "charts_index.json", {"charts": charts, "chart_dir": str(CHARTS_DIR), "chart_count": len(charts), **meta})
_write_json(ANALYTICS_DIR / "dataset_analytics_bundle.json", {**stats, "charts": charts})

print("JSON files written to", ANALYTICS_DIR)
print("metadata:", json.dumps(meta, ensure_ascii=False, indent=2))


In [ ]:
# Cell 9 — Zip và tải về máy local
ZIP_NAME = "vietnews_analytics_colab.zip"
zip_path = WORKDIR / ZIP_NAME

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ANALYTICS_DIR.glob("*.json"):
        zf.write(f, f"analytics/{f.name}")
    for f in CHARTS_DIR.glob("*.png"):
        zf.write(f, f"charts/{f.name}")

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Created {zip_path} ({size_mb:.1f} MB)")
print("Files:", len(list(ANALYTICS_DIR.glob('*.json'))), "JSON,", len(list(CHARTS_DIR.glob('*.png'))), "PNG")

if IS_COLAB:
    from google.colab import files
    files.download(str(zip_path))
    print("Download started — lưu file và chạy trên máy local:")
    print("  python scripts/import_colab_analytics.py vietnews_analytics_colab.zip")
else:
    print("Không phải Colab — zip tại:", zip_path)
